In [2]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

def parse_mean_std(mean_std_str):
    """Parse '0.1234 ± 0.0567' format"""
    if pd.isna(mean_std_str) or mean_std_str == '':
        return np.nan, np.nan
    parts = str(mean_std_str).split(' ± ')
    if len(parts) == 2:
        return float(parts[0]), float(parts[1])
    return np.nan, np.nan

def calculate_cohens_d(mean1, std1, n1, mean2, std2, n2):
    """Calculate Cohen's d effect size"""
    if n1 <= 1 or n2 <= 1 or std1 == 0 or std2 == 0:
        return np.nan
    
    # Pooled standard deviation
    pooled_std = np.sqrt(((n1 - 1) * std1**2 + (n2 - 1) * std2**2) / (n1 + n2 - 2))
    
    if pooled_std == 0:
        return np.nan
    
    # Cohen's d
    cohens_d = (mean1 - mean2) / pooled_std
    return cohens_d

# def interpret_effect_size(d):
#     """Interpret Cohen's d according to Cohen's conventions"""
#     abs_d = abs(d)
#     if abs_d < 0.2:
#         return "negligible"
#     elif abs_d < 0.5:
#         return "small"
#     elif abs_d < 0.8:
#         return "medium"
#     else:
#         return "large"

def interpret_effect_size_nlp(d):
    """NLP-focused interpretation of Cohen's d"""
    abs_d = abs(d)
    if abs_d < 0.1:
        return "negligible"
    elif abs_d < 0.3:
        return "small"
    elif abs_d < 0.6:
        return "medium"
    else:
        return "large"

def create_effect_size_analysis(base_dir, output_dir, 
                               min_occurrence_count=3,  # Relaxed threshold
                               min_effect_size=0.2,     # Include small+ effects
                               total_iterations=13):
    """
    Aggregate features based on effect size analysis:
    1. Calculate Cohen's d for each feature in each iteration
    2. Include features with meaningful effect sizes (≥0.2) in multiple iterations
    3. Less strict occurrence requirements
    """
    stances = ["All_Stances", "FAVOR", "AGAINST", "NONE"]
    models = ["Logistic Regression", "XGBoost"]
    
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Configuration - EFFECT SIZE ANALYSIS:")
    print(f"  1. Calculate Cohen's d for each feature in each iteration")
    print(f"  2. Include features with effect size ≥{min_effect_size} in ≥{min_occurrence_count} iterations")
    print(f"  3. Cohen's d interpretation: <0.2=negligible, 0.2-0.5=small, 0.5-0.8=medium, >0.8=large")
    print(f"  Total iterations: {total_iterations}")
    
    summary_stats = []
    
    for stance in stances:
        print(f"\nProcessing {stance}...")
        
        # Step 1: Collect all features from all iterations WITH effect size calculation
        all_feature_data = {}
        
        for i in range(1, total_iterations + 1):
            file_path = os.path.join(base_dir, f"{i:02d}_iteration", f"top_features_{stance}.csv")
            if os.path.exists(file_path):
                df = pd.read_csv(file_path)
                print(f"  Loaded iteration {i:02d}: {len(df)} features")
                
                for _, row in df.iterrows():
                    clean_feature = str(row['feature']).replace('*', '')
                    
                    if clean_feature not in all_feature_data:
                        all_feature_data[clean_feature] = []
                    
                    # Parse mean/std for effect size calculation
                    correct_mean, correct_std = parse_mean_std(row.get('correct_mean_std', ''))
                    incorrect_mean, incorrect_std = parse_mean_std(row.get('incorrect_mean_std', ''))
                    
                    # Calculate Cohen's d (assuming equal sample sizes for now)
                    # You might want to extract actual sample sizes from your original data
                    estimated_n = 100  # Placeholder - ideally get from your data
                    cohens_d = calculate_cohens_d(correct_mean, correct_std, estimated_n,
                                                incorrect_mean, incorrect_std, estimated_n)
                    
                    feature_occurrence = {
                        'iteration': i,
                        'is_significant': '*' in str(row['feature']),
                        'coefficients': {},
                        'correct_mean': correct_mean,
                        'correct_std': correct_std,
                        'incorrect_mean': incorrect_mean,
                        'incorrect_std': incorrect_std,
                        'cohens_d': cohens_d,
                        'effect_size_category': interpret_effect_size(cohens_d) if not np.isnan(cohens_d) else 'unknown',
                        'correct_mean_std': row.get('correct_mean_std', ''),
                        'incorrect_mean_std': row.get('incorrect_mean_std', '')
                    }
                    
                    # Store coefficients/importances
                    for model in models:
                        if (model in df.columns and 
                            pd.notna(row[model]) and 
                            row[model] != '' and 
                            row[model] != 0):
                            feature_occurrence['coefficients'][model] = float(row[model])
                    
                    all_feature_data[clean_feature].append(feature_occurrence)
            else:
                print(f"  Warning: Missing file {file_path}")
        
        print(f"  Total features found: {len(all_feature_data)}")
        
        # Step 2: Apply effect size-based filtering
        final_features = []
        
        for feature_name, occurrences in all_feature_data.items():
            # Aggregate data across occurrences
            aggregated_data = {
                'feature_name': feature_name,
                'total_occurrences': len(occurrences),
                'coefficients': {model: [] for model in models},
                'coef_occurrences': {model: 0 for model in models},
                'significances': [],
                'cohens_d_values': [],
                'meaningful_effect_count': 0,  # Count of |Cohen's d| >= min_effect_size
                'large_effect_count': 0,       # Count of |Cohen's d| >= 0.8
                'correct_means': [],
                'incorrect_means': [],
                'correct_stds': [],
                'incorrect_stds': []
            }
            
            # Collect data from all occurrences
            for occurrence in occurrences:
                aggregated_data['significances'].append(occurrence['is_significant'])
                
                # Collect effect sizes
                if not np.isnan(occurrence['cohens_d']):
                    aggregated_data['cohens_d_values'].append(occurrence['cohens_d'])
                    
                    # Count meaningful effects
                    if abs(occurrence['cohens_d']) >= min_effect_size:
                        aggregated_data['meaningful_effect_count'] += 1
                    if abs(occurrence['cohens_d']) >= 0.8:
                        aggregated_data['large_effect_count'] += 1
                
                # Collect coefficients
                for model in models:
                    if model in occurrence['coefficients']:
                        aggregated_data['coefficients'][model].append(occurrence['coefficients'][model])
                        aggregated_data['coef_occurrences'][model] += 1
                
                # Collect means/stds
                if not np.isnan(occurrence['correct_mean']):
                    aggregated_data['correct_means'].append(occurrence['correct_mean'])
                    aggregated_data['correct_stds'].append(occurrence['correct_std'])
                    aggregated_data['incorrect_means'].append(occurrence['incorrect_mean'])
                    aggregated_data['incorrect_stds'].append(occurrence['incorrect_std'])
            
            # FILTER 1: Must have meaningful effect size in at least min_occurrence_count iterations
            if aggregated_data['meaningful_effect_count'] < min_occurrence_count:
                print(f"    Skipping {feature_name}: meaningful effect size in only {aggregated_data['meaningful_effect_count']}/{len(occurrences)} iterations (need ≥{min_occurrence_count})")
                continue
            
            # FILTER 2: Must have some coefficient data (relaxed - either LR or XGBoost)
            lr_coeffs = aggregated_data['coefficients']['Logistic Regression']
            xgb_coeffs = aggregated_data['coefficients']['XGBoost']
            
            if not lr_coeffs and not xgb_coeffs:
                print(f"    Skipping {feature_name}: no coefficient data")
                continue
            
            # Calculate final values
            median_cohens_d = np.median(aggregated_data['cohens_d_values']) if aggregated_data['cohens_d_values'] else np.nan
            final_lr_coef = np.median(lr_coeffs) if lr_coeffs else np.nan
            final_xgb_importance = np.median(xgb_coeffs) if xgb_coeffs else np.nan
            
            # Check LR sign consistency (if we have LR coefficients)
            lr_sign_consistent = True
            lr_sign = 'none'
            if lr_coeffs:
                lr_signs = [1 if x > 0 else -1 if x < 0 else 0 for x in lr_coeffs]
                lr_non_zero_signs = [s for s in lr_signs if s != 0]
                if len(set(lr_non_zero_signs)) > 1:
                    lr_sign_consistent = False
                elif lr_non_zero_signs:
                    lr_sign = 'positive' if lr_non_zero_signs[0] > 0 else 'negative'
            
            # Calculate aggregated statistics
            agg_correct_mean = np.mean(aggregated_data['correct_means']) if aggregated_data['correct_means'] else np.nan
            agg_correct_std = np.mean(aggregated_data['correct_stds']) if aggregated_data['correct_stds'] else np.nan
            agg_incorrect_mean = np.mean(aggregated_data['incorrect_means']) if aggregated_data['incorrect_means'] else np.nan
            agg_incorrect_std = np.mean(aggregated_data['incorrect_stds']) if aggregated_data['incorrect_stds'] else np.nan
            
            # Count significance occurrences
            num_significant = sum(aggregated_data['significances'])
            
            # Create final entry
            final_features.append({
                'feature': feature_name,
                'Logistic Regression': final_lr_coef if not np.isnan(final_lr_coef) else '',
                'XGBoost': final_xgb_importance if not np.isnan(final_xgb_importance) else '',
                'correct_mean_std': f"{agg_correct_mean:.4f} ± {agg_correct_std:.4f}" if not np.isnan(agg_correct_mean) else '',
                'incorrect_mean_std': f"{agg_incorrect_mean:.4f} ± {agg_incorrect_std:.4f}" if not np.isnan(agg_incorrect_mean) else '',
                'median_cohens_d': round(median_cohens_d, 3) if not np.isnan(median_cohens_d) else '',
                'effect_size_category': interpret_effect_size(median_cohens_d) if not np.isnan(median_cohens_d) else '',
                'meaningful_effect_count': aggregated_data['meaningful_effect_count'],
                'large_effect_count': aggregated_data['large_effect_count'],
                'lr_coef_count': len(lr_coeffs),
                'xgb_coef_count': len(xgb_coeffs),
                'num_significant': num_significant,
                'significance_rate': f"{num_significant}/{len(occurrences)}",
                'lr_sign': lr_sign,
                'lr_sign_consistent': lr_sign_consistent,
                'avg_abs_cohens_d': round(np.mean([abs(d) for d in aggregated_data['cohens_d_values']]), 3) if aggregated_data['cohens_d_values'] else 0
            })
        
        # Step 3: Sort by effect size magnitude, then by meaningful effect count
        final_features.sort(key=lambda x: (x['avg_abs_cohens_d'], x['meaningful_effect_count']), reverse=True)
        
        # Remove helper columns for final output
        final_features_clean = []
        for feature in final_features:
            clean_feature = {k: v for k, v in feature.items() 
                           if k not in ['lr_coef_count', 'xgb_coef_count', 'meaningful_effect_count', 'large_effect_count', 
                                      'num_significant', 'significance_rate', 'lr_sign', 'lr_sign_consistent', 'avg_abs_cohens_d']}
            final_features_clean.append(clean_feature)
        
        # Step 4: Save final aggregated file
        final_df = pd.DataFrame(final_features_clean)
        output_file = os.path.join(output_dir, f"final_top_features_{stance}.csv")
        final_df.to_csv(output_file, index=False)
        
        # Step 5: Save detailed info file with effect size analysis
        detailed_df = pd.DataFrame(final_features)
        detailed_file = os.path.join(output_dir, f"detailed_features_{stance}.csv")
        detailed_df.to_csv(detailed_file, index=False)
        
        # Collect summary statistics
        total_features = len(all_feature_data)
        final_count = len(final_features)
        large_effects = len([f for f in final_features if f['effect_size_category'] == 'large'])
        medium_effects = len([f for f in final_features if f['effect_size_category'] == 'medium'])
        small_effects = len([f for f in final_features if f['effect_size_category'] == 'small'])
        
        summary_stats.append({
            'Stance': stance,
            'Total_Features': total_features,
            'Final_Features': final_count,
            'Large_Effects': large_effects,
            'Medium_Effects': medium_effects,
            'Small_Effects': small_effects,
            'Avg_Effect_Size': round(np.mean([f['avg_abs_cohens_d'] for f in final_features]) if final_features else 0, 3),
            'Success_Rate': f"{final_count}/{total_features}",
            'Success_Percentage': f"{100*final_count/total_features:.1f}%" if total_features > 0 else "0%"
        })
        
        print(f"  Final features with meaningful effect sizes: {final_count}")
        print(f"  Large effects (d≥0.8): {large_effects}")
        print(f"  Medium effects (0.5≤d<0.8): {medium_effects}")
        print(f"  Small effects (0.2≤d<0.5): {small_effects}")
        print(f"  Average effect size: {round(np.mean([f['avg_abs_cohens_d'] for f in final_features]) if final_features else 0, 3)}")
        print(f"  Success rate: {100*final_count/total_features:.1f}%")
        print(f"  Saved to: {output_file}")
        print(f"  Detailed info saved to: {detailed_file}")
    
    # Save summary statistics
    summary_df = pd.DataFrame(summary_stats)
    summary_file = os.path.join(output_dir, f"aggregation_summary_effect_size_{min_effect_size}.csv")
    summary_df.to_csv(summary_file, index=False)
    
    print(f"\n" + "="*80)
    print(f"AGGREGATION SUMMARY - EFFECT SIZE ANALYSIS (Cohen's d ≥{min_effect_size})")
    print("="*80)
    print(summary_df.to_string(index=False))
    print(f"\nSummary saved to: {summary_file}")
    
    return summary_df

# Main execution
if __name__ == '__main__':
    base_dir = ""
    output_dir = "important_features_effect_size"
    
    # CONFIGURABLE PARAMETERS
    MIN_OCCURRENCE_COUNT = 7     # Feature must have meaningful effect in at least 3 iterations
    MIN_EFFECT_SIZE = 0.2        # Small effect size threshold
    TOTAL_ITERATIONS = 13
    
    print("Creating final aggregated feature files with EFFECT SIZE analysis...")
    print("Criteria:")
    print(f"1. Cohen's d effect size ≥{MIN_EFFECT_SIZE} in ≥{MIN_OCCURRENCE_COUNT} iterations")
    print("2. Relaxed coefficient requirements (LR OR XGBoost)")
    print("3. Cohen's d interpretation: <0.2=negligible, 0.2-0.5=small, 0.5-0.8=medium, >0.8=large")
    
    summary = create_effect_size_analysis(
        base_dir=base_dir,
        output_dir=output_dir,
        min_occurrence_count=MIN_OCCURRENCE_COUNT,
        min_effect_size=MIN_EFFECT_SIZE,
        total_iterations=TOTAL_ITERATIONS
    )
    
    print(f"\n✅ Effect size analysis completed!")
    print(f"📁 Created files:")
    print(f"   - final_top_features_*.csv (4 files) - features with meaningful effect sizes")
    print(f"   - detailed_features_*.csv (4 files with Cohen's d analysis)")
    print(f"   - aggregation_summary_effect_size_{MIN_EFFECT_SIZE}.csv")

Creating final aggregated feature files with EFFECT SIZE analysis...
Criteria:
1. Cohen's d effect size ≥0.2 in ≥7 iterations
2. Relaxed coefficient requirements (LR OR XGBoost)
3. Cohen's d interpretation: <0.2=negligible, 0.2-0.5=small, 0.5-0.8=medium, >0.8=large
Configuration - EFFECT SIZE ANALYSIS:
  1. Calculate Cohen's d for each feature in each iteration
  2. Include features with effect size ≥0.2 in ≥7 iterations
  3. Cohen's d interpretation: <0.2=negligible, 0.2-0.5=small, 0.5-0.8=medium, >0.8=large
  Total iterations: 13

Processing All_Stances...
  Loaded iteration 01: 44 features
  Loaded iteration 02: 44 features
  Loaded iteration 03: 44 features
  Loaded iteration 04: 44 features
  Loaded iteration 05: 44 features
  Loaded iteration 06: 44 features
  Loaded iteration 07: 44 features
  Loaded iteration 08: 44 features
  Loaded iteration 09: 44 features
  Loaded iteration 10: 44 features
  Loaded iteration 11: 44 features
  Loaded iteration 12: 44 features
  Loaded iterat